PySpark Training


---



---



In [3]:
print('hello')

hello


In [4]:
!pip install pyspark

In [75]:
from google.colab import drive
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnull, unix_timestamp, round

In [ ]:

drive.mount('/content/drive')

In [68]:
pyspark.__version__

'4.0.2'

In [9]:
import requests

url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
output_file = "taxi_zone_lookup.csv"

response = requests.get(url)
response.raise_for_status()  # raises error if download fails

with open(output_file, "wb") as f:
    f.write(response.content)

print("Downloaded successfully:", output_file)


Downloaded successfully: taxi_zone_lookup.csv


In [11]:
response1 = requests.get('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet')
response1.raise_for_status()

with open('yellow_tripdata_2025-01.parquet', "wb") as g:
  g.write(response1.content)

print("Downloaded successfully:", "yellow_tripdata_2025-01.parquet")

Downloaded successfully: yellow_tripdata_2025-01.parquet


In [12]:
response2 = requests.get('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-02.parquet')
response2.raise_for_status()

with open('yellow_tripdata_2025-02.parquet', "wb") as h:
  h.write(response2.content)

print("Downloaded successfully:", "yellow_tripdata_2025-02.parquet")

Downloaded successfully: yellow_tripdata_2025-02.parquet


In [21]:
%%sh
pwd
ls /content/pyspark_training

/content
taxi_zone_lookup.csv
yellow_tripdata_2025-01.parquet
yellow_tripdata_2025-02.parquet


In [23]:
os.path.isfile('/content/pyspark_training/taxi_zone_lookup.csv')

True

In [26]:
os.path.isdir('/content/pyspark_training')

True

In [27]:
spark = SparkSession.builder.getOrCreate()

In [28]:
spark

In [31]:
january_file = '/content/pyspark_training/yellow_tripdata_2025-01.parquet'
df = spark.read.parquet(january_file)

In [34]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

In [36]:
# now of rows
df.count()

3475226

In [37]:
df.schema.names

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [39]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [44]:
df.describe(['passenger_count', 'trip_distance', 'total_amount']).show()

+-------+------------------+------------------+------------------+
|summary|   passenger_count|     trip_distance|      total_amount|
+-------+------------------+------------------+------------------+
|  count|           2935077|           3475226|           3475226|
|   mean|1.2978589658806226|5.8551261788432925|25.611291697295062|
| stddev| 0.750750275480471|  564.601599634634| 463.6584784502283|
|    min|                 0|               0.0|            -901.0|
|    max|                 9|         276423.57|         863380.37|
+-------+------------------+------------------+------------------+



In [48]:
df.select('passenger_count').show()

+---------------+
|passenger_count|
+---------------+
|              1|
|              1|
|              1|
|              3|
|              3|
|              2|
|              0|
|              0|
|              0|
|              1|
|              1|
|              1|
|              3|
|              1|
|              1|
|              3|
|              1|
|              1|
|              1|
|              2|
+---------------+
only showing top 20 rows


In [50]:
df.select(col('passenger_count')).show()

+---------------+
|passenger_count|
+---------------+
|              1|
|              1|
|              1|
|              3|
|              3|
|              2|
|              0|
|              0|
|              0|
|              1|
|              1|
|              1|
|              3|
|              1|
|              1|
|              3|
|              1|
|              1|
|              1|
|              2|
+---------------+
only showing top 20 rows


In [51]:
df.sort('total_amount', ascending=False).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-20 12:07:18|  2025-01-20 12:12:42|              1|          1.6|         1|                 N|         138|    

In [52]:
df.sort(['total_amount', 'passenger_count'], ascending = [False, False]).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-20 12:07:18|  2025-01-20 12:12:42|              1|          1.6|         1|                 N|         138|    

In [53]:
df.filter('Airport_fee > 0').show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:51:41|  2025-01-01 01:06:26|              1|          7.2|         1|                 N|         132|    

In [67]:
df.filter('Airport_fee = 0' and 'passenger_count = 1')\
.select(['trip_distance', 'total_amount'])\
.sort('trip_distance', ascending=False)\
.show()

+-------------+------------+
|trip_distance|total_amount|
+-------------+------------+
|      44730.3|        45.0|
|      44684.1|       54.94|
|      33588.9|        30.0|
|      11187.2|       61.94|
|      4020.04|      101.05|
|      2001.95|       19.35|
|      1847.61|       43.37|
|      1472.37|       20.58|
|        265.9|      139.14|
|       255.33|     2506.71|
|       206.45|      243.32|
|        199.3|        19.0|
|       188.88|      1311.7|
|        181.9|       39.94|
|       150.11|      501.75|
|        148.3|      549.91|
|       143.54|      969.05|
|        139.7|      452.75|
|        133.3|       896.5|
|       132.27|     -865.39|
+-------------+------------+
only showing top 20 rows


In [70]:
df.filter(isnull(col('fare_amount'))).count()

0

In [71]:
df.filter(isnull(col('passenger_count'))).count()

540149

In [72]:
df1 = df.fillna({'passenger_count': 1})

In [73]:
df1.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

In [74]:
df1.filter(isnull(col('passenger_count'))).count()

0

In [82]:
df1 = df.withColumn('trip_duration_minutes',
                    round((unix_timestamp('tpep_dropoff_datetime') -  unix_timestamp('tpep_pickup_datetime'))/60, 1))\
                    .sort('trip_duration_minutes', ascending=False)\
                    .show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|trip_duration_minutes|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+
|       2| 2025-01-21 12:23:41|  2025-01-25 10:10:00|           

In [ ]:
# withColumnRenamed
# drop('col1', 'col2')